# Step 3 — 正式训练数据生成与标注

在 Pilot 通过后生成 `2,000 prompts × 2 = 4,000 trajectories`，并写入 correctness 标签。

本阶段再次检查截断和标签平衡，防止规模扩大后数据分布发生变化。

In [ ]:
# @title Step 03.1 — 初始化运行环境
from pathlib import Path
import gc
import importlib.util
import json
import os
import shutil
import subprocess
import sys

REPO = Path("/content/ZIP-RC-Colab")
ZIP_PY = Path("/content/mamba/envs/zip/bin/python")
REPO_URL = "https://github.com/wtree101/ZIP-RC-Colab.git"
REPO_BRANCH = "main"
SYNC_REPO = True

if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO)], check=True)
elif SYNC_REPO:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

if not ZIP_PY.exists():
    raise FileNotFoundError(
        f"未找到 {ZIP_PY}。请先建立 ZIP-RC 的 mamba 环境，再重新运行本 Notebook。"
    )

kernel_required = ["torch", "numpy", "pandas", "pyarrow", "matplotlib", "sklearn", "psutil"]
kernel_missing = [name for name in kernel_required if importlib.util.find_spec(name) is None]
if kernel_missing:
    raise ModuleNotFoundError(f"Colab kernel 缺少可视化依赖: {kernel_missing}")

env_check = subprocess.run(
    [
        str(ZIP_PY),
        "-c",
        (
            "import importlib.util, json; "
            "mods=['torch','vllm','transformers','datasets','pandas','pyarrow','tqdm']; "
            "print(json.dumps([m for m in mods if importlib.util.find_spec(m) is None]))"
        ),
    ],
    check=True,
    capture_output=True,
    text=True,
)
env_missing = json.loads(env_check.stdout.strip())
if env_missing:
    raise ModuleNotFoundError(f"zip mamba 环境缺少依赖: {env_missing}")

os.environ["ZIPRC_PYTHON"] = str(ZIP_PY)
os.environ["PATH"] = f"{Path.home() / '.local/bin'}{os.pathsep}{os.environ['PATH']}"

import matplotlib.pyplot as plt
import pandas as pd
import psutil
import torch
from IPython.display import display

sys.path.insert(0, str(REPO / "notebooks"))
from ziprc_notebook_utils import (
    gate,
    gate_frame,
    load_config,
    model_artifacts_exist,
    progress_frame,
    read_jsonl,
    require_columns,
    rolling_edges,
    run_repo,
    save_stage_report,
)

print("Repository:", REPO)
print("ZIP Python:", ZIP_PY)

CONFIG = load_config(REPO)
print("Experiment:", CONFIG["experiment_name"])


In [ ]:
# @title Step 03.2 — 查看持久化进度
runtime_progress = progress_frame(REPO)
print("持久化进度快照：")
display(runtime_progress if not runtime_progress.empty else pd.DataFrame([{"状态": "尚无进度记录"}]))

In [ ]:
# @title Step 03.3 — 生成并评分正式训练数据
RUN_STAGE = True
full_path = REPO / CONFIG["paths"]["full"]
full_grader_metrics = REPO / "artifacts/metrics/full_grader.json"
if RUN_STAGE:
    run_repo(
        REPO,
        "python3", "src/generate_ziprc_rollouts.py",
        "--model", CONFIG["model_id"], "--dataset", CONFIG["dataset"],
        "--split", CONFIG["split"], "--prompt-column", CONFIG["prompt_column"],
        "--answer-column", CONFIG["answer_column"], "--out", full_path,
        "--max-num-prompts", CONFIG["training_prompts"],
        "--thinking-samples", 0, "--non-thinking-samples", CONFIG["training_rollouts_per_prompt"],
        "--temperature", CONFIG["temperature"], "--min-p", CONFIG["min_p"],
        "--max-model-len", CONFIG["generation_max_model_len"],
        "--max-new-tokens", CONFIG["max_output_tokens"],
        "--max-num-seqs", CONFIG["max_num_seqs"], "--dtype", CONFIG["dtype"],
        "--dp-size", 1, "--tp-size", 1,
    )
    run_repo(
        REPO,
        "python3", "src/evaluate_and_label_rollouts.py",
        "--data", full_path, "--model", CONFIG["grader_model_id"],
        "--tensor-parallel-size", 1, "--gpu-memory-utilization", CONFIG["gpu_memory_utilization"],
        "--max-model-len", CONFIG["grader_max_model_len"], "--max-num-seqs", CONFIG["max_num_seqs"],
        "--dtype", CONFIG["dtype"], "--output-json", full_grader_metrics,
    )

In [ ]:
# @title Step 03.4 — 检查长度、标签与题目难度
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

df = pd.read_parquet(full_path)
expected = int(CONFIG["training_prompts"]) * int(CONFIG["training_rollouts_per_prompt"])
accuracy = float(df["correct"].mean())
finished_rate = float(df["finished"].mean())
cap_rate = float((df["length"] >= CONFIG["max_output_tokens"]).mean())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df["length"], bins=40, color="#4c78a8")
axes[0].axvline(CONFIG["max_output_tokens"], color="#e45756", linestyle="--")
axes[0].set(title="Full-data response length", xlabel="tokens")
df["correct"].value_counts().sort_index().plot.bar(ax=axes[1], color=["#e45756", "#49beaa"])
axes[1].set_title("Correct / incorrect samples")
per_prompt = df.groupby("prompt_idx").agg(correct_rate=("correct", "mean"), mean_length=("length", "mean"))
axes[2].scatter(per_prompt["mean_length"], per_prompt["correct_rate"], s=10, alpha=.35)
axes[2].set(title="Per-prompt difficulty", xlabel="mean output tokens", ylabel="correct rate")
plt.tight_layout()
plt.show()

checks = [
    gate("样本数完整", len(df) == expected, f"{len(df)}/{expected}"),
    gate("Prompt 数完整", df["prompt_idx"].nunique() == CONFIG["training_prompts"], f"{df['prompt_idx'].nunique()} prompts"),
    gate("Finished rate ≥95%", finished_rate >= .95, f"{finished_rate:.1%}", kind="scientific"),
    gate("截断 <5%", cap_rate < .05, f"{cap_rate:.1%}", kind="scientific"),
    gate("正负标签可学习", df["correct"].nunique() == 2 and df["correct"].value_counts(normalize=True).min() >= .10, f"accuracy={accuracy:.1%}", kind="scientific"),
]
display(gate_frame(checks))
save_stage_report(REPO, "03_training_data", checks, {"rows": len(df), "prompts": int(df['prompt_idx'].nunique()), "accuracy": accuracy, "finished_rate": finished_rate, "cap_rate": cap_rate})

In [ ]:
# @title Step 03.5 — 检查 grader 有效输出
import warnings

finished_labels = df.loc[df["finished"]]
if "grader_valid" in finished_labels.columns:
    grader_valid = finished_labels["grader_valid"].fillna(False).astype(bool)
else:
    grader_valid = pd.Series(False, index=finished_labels.index)
grader_valid_rate = float(grader_valid.mean()) if len(grader_valid) else 0.0

fig, ax = plt.subplots(figsize=(5, 3))
grader_valid.value_counts().reindex([True, False], fill_value=0).plot.bar(
    ax=ax, color=["#49beaa", "#e45756"]
)
ax.set(title=f"Grader validity ({grader_valid_rate:.1%} valid)", xlabel="grader_valid", ylabel="rows")
plt.tight_layout()
plt.show()

grader_check = gate(
    "Finished rows 的 grader 输出全部有效",
    grader_valid_rate == 1.0,
    f"valid={int(grader_valid.sum())}/{len(grader_valid)}",
)
checks = [*checks, grader_check]
display(gate_frame([grader_check]))
save_stage_report(
    REPO,
    "03_training_data",
    checks,
    {
        "rows": len(df),
        "prompts": int(df["prompt_idx"].nunique()),
        "accuracy": accuracy,
        "finished_rate": finished_rate,
        "cap_rate": cap_rate,
        "grader_valid_rate": grader_valid_rate,
    },
)
if grader_valid_rate < 1.0:
    warnings.warn(
        "存在无效 grader 输出；批处理已跑完，但这些行的 correct=False 只是回退值。",
        RuntimeWarning,
    )